# **This notebook implements a EEG Auditory Attention Detectiom (AAD) system using Domain Adversarial Learning. The goal is to predict which speaker a subject is attending to while learning subject-invariant features through domain adaptation.**

# **Importing Libraries**

In [ ]:
# Standard Library Imports
import os            # OS-level operations (paths, directory handling)
import math          # Mathematical functions
import shutil        # High-level file operations (copying, moving, deleting)

# Data Handling & Processing
import numpy as np               # Numerical computations and array operations
import pandas as pd              # Data manipulation and analysis
import h5py                      # Handling HDF5 file formats

# Progress Visualization
from tqdm import tqdm            # Progress bars for loops

# Plotting & Visualization
import matplotlib.pyplot as plt  # Plotting and visualization tools


# PyTorch Machine Learning Stack
import torch                     # Core PyTorch library
import torch.nn as nn            # Neural network layers and utilities
import torch.optim as optim      # Optimization algorithms (SGD, Adam, etc.)
from torch.utils.data import (   # Dataset and DataLoader utilities
    DataLoader,
    Dataset
)

# **Function to load h5 files**

In [ ]:
def load_h5_dataset(file_path):
    """
    Load dataset, labels, and subject IDs from an HDF5 (.h5) file.

    Parameters
    ----------
    file_path : str
        Path to the HDF5 file containing 'data', 'label', and 'sub_id' datasets.

    Returns
    -------
    data : np.ndarray
        The input data stored in the HDF5 file.
    label : np.ndarray
        Corresponding labels for each data sample.
    subjects : np.ndarray
        Subject IDs associated with each data sample.
    """

    # Open the .h5 file in read-only mode to ensure safe, non-destructive access
    with h5py.File(file_path, 'r') as f:
        # Load arrays stored in the HDF5 datasets
        data = np.array(f['data'])       # Main feature/data array
        label = np.array(f['label'])     # Labels or targets for each sample
        subjects = np.array(f['sub_id']) # Subject identifier for each data entry

    # Return loaded components as NumPy arrays
    return data, label, subjects


# **define pytorch class for data**

In [ ]:
class CustomDatasets(Dataset):
    """
    A PyTorch Dataset wrapper for handling data, labels, and subject IDs.

    This class allows data to be easily fed into a DataLoader for batching,
    shuffling, and parallel loading during training or evaluation.
    """

    def __init__(self, data, labels, subjects):
        """
        Initialize the dataset.

        Parameters
        ----------
        data : array-like
            Input feature data, typically a NumPy array.
        labels : array-like
            Integer class labels for each sample.
        subjects : array-like
            Subject IDs corresponding to each sample (useful for subject-level splits).
        """
        self.data = data
        self.labels = labels
        self.subjects = subjects

    def __len__(self):
        """
        Return the total number of samples in the dataset.
        """
        return len(self.labels)

    def __getitem__(self, index):
        """
        Retrieve a single sample by index.

        Returns
        -------
        x : torch.Tensor
            Feature tensor for the given index.
        y : torch.Tensor
            Label tensor for the given index.
        s : torch.Tensor
            Subject ID tensor for the given index.
        """
        # Convert selected sample, label, and subject ID into PyTorch tensors
        x = torch.tensor(self.data[index], dtype=torch.float32)
        y = torch.tensor(self.labels[index], dtype=torch.long)
        s = torch.tensor(self.subjects[index], dtype=torch.long)

        return x, y, s


# **Model architecture**

In [ ]:
# Gradient Reversal Layer (GRL)
# Used in Domain-Adversarial Neural Networks (DANN)
# Reverses gradient during backprop to encourage domain invariance
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        """
        Forward pass: acts as identity function.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor.
        lambd : float
            Reversal strength; used in the backward pass.
        """
        ctx.lambd = lambd                 # Store λ for backward pass
        return x.view_as(x)               # Identity operation

    @staticmethod
    def backward(ctx, grad_output):
        """
        Backward pass: multiply gradient by -λ.
        This forces the feature extractor to learn domain-invariant features.
        """
        return grad_output.neg() * ctx.lambd, None


def grad_reverse(x, lambd):
    """Convenience wrapper for applying the GRL."""
    return GradReverse.apply(x, lambd)


# Token Embedding Module
# Converts raw EEG input into token embeddings suitable for LSTM
class TokenEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        """
        Parameters
        ----------
        c_in : int
            Number of EEG input channels.
        d_model : int
            Embedding dimension for each time step.
        """
        super(TokenEmbedding, self).__init__()

        # First embedding layer:
        # 1 → (d_model*4) channels using temporal convolution
        self.embed_layer = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=d_model * 4,
                kernel_size=(1, 8),
                padding='same'
            ),
            nn.BatchNorm2d(d_model * 4),
            nn.GELU()
        )

        # Second embedding layer:
        # (d_model*4) → d_model channels using spatial convolution across channels
        self.embed_layer2 = nn.Sequential(
            nn.Conv2d(
                in_channels=d_model * 4,
                out_channels=d_model,
                kernel_size=(c_in, 1),   # Span all channels
                padding='valid'
            ),
            nn.BatchNorm2d(d_model),
            nn.GELU()
        )

    def forward(self, x):
        """
        x : (B, C, T)
        Returns token embeddings: (B, T, d_model)
        """
        x = x.unsqueeze(1)            # (B, 1, C, T)
        x = self.embed_layer(x)       # (B, d_model*4, C, T)
        x = self.embed_layer2(x)      # (B, d_model, 1, T)
        x = x.squeeze(2)              # (B, d_model, T)
        x = x.permute(0, 2, 1)        # (B, T, d_model)
        return x


# tokenembedding + LSTM + DANN Model
# - Token embedding through convolution
# - LSTM sequence model
# - Task classifier (main task)
# - Domain classifier (subject classifier) with GRL
class DARNet_LSTM_DANN(nn.Module):
    def __init__(self, c_in=32, d_model=16, hidden=64,
                 num_classes=2, num_subjects=30):
        """
        Parameters
        ----------
        c_in : int
            Number of EEG input channels.
        d_model : int
            Embedding dimension per token.
        hidden : int
            Hidden size of the LSTM.
        num_classes : int
            Number of output task classes.
        num_subjects : int
            Number of domain classes (e.g., subjects).
        """
        super().__init__()

        # Embed raw data into feature tokens
        self.token_embed = TokenEmbedding(c_in, d_model)

        # Bidirectional LSTM for temporal feature extraction
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Task classifier (main AAD task)
        self.classifier = nn.Linear(hidden * 2, num_classes)

        # Domain classifier (subject prediction)
        # Used via gradient reversal for adversarial training
        self.domain_classifier = nn.Sequential(
            nn.Linear(hidden * 2, 64),
            nn.ReLU(),
            nn.Linear(64, num_subjects)
        )

    def forward(self, x, lambd=0.0):
        """
        Forward pass with optional gradient reversal.

        Parameters
        ----------
        x : (B, C, T)
            Input EEG batch.
        lambd : float
            Lambda for gradient reversal (0 during evaluation).
        """
        # 1. Token embedding
        emb = self.token_embed(x)                          # (B, T, d_model)

        # 2. Sequence modeling
        features, _ = self.lstm(emb)                       # (B, T, hidden*2)

        # 3. Global average pooling across time
        feat = features.mean(dim=1)                        # (B, hidden*2)

        # ----- Task prediction -----
        logits_task = self.classifier(feat)

        # ----- Domain prediction (with gradient reversal) -----
        rev = grad_reverse(feat, lambd)                    # Reverse gradients
        logits_domain = self.domain_classifier(rev)

        return logits_task, logits_domain


# **Function for training**

In [ ]:
def train_model(model, train_loader, val_loader, epochs=100, lr=5e-4,
                weight_decay=3e-4, device="cuda"):
    """
    Train a DARNet-LSTM-DANN model with adversarial domain adaptation.

    Parameters
    ----------
    model : nn.Module
        The neural network model.
    train_loader : DataLoader
        DataLoader containing training batches.
    val_loader : DataLoader
        DataLoader containing validation batches.
    epochs : int
        Number of training epochs.
    lr : float
        Learning rate for optimizer.
    weight_decay : float
        Weight decay (L2 regularization).
    device : str
        "cuda" or "cpu".

    Returns
    -------
    None
    (Saves best model as "best_model.pth")
    """

    # Move model to device (GPU or CPU)
    model = model.to(device)

    # AdamW optimizer for stable training
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Cosine learning rate annealing scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=10,      # Period of cosine curve
        eta_min=1e-6   # Minimum learning rate
    )

    # Loss functions: task classification + domain classification
    criterion_task = nn.CrossEntropyLoss()
    criterion_domain = nn.CrossEntropyLoss()

    # Tracking metrics
    best_val_acc = 0
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    # Training loop
    for epoch in range(epochs):
        model.train()          # Enable training mode
        running_loss = 0       # Accumulate epoch loss
        correct = 0            # Count correct predictions
        total = 0              # Number of samples

        # DANN lambda increases from 0 → 1 during training
        # Smooth schedule recommended by the DANN paper
        lambd = 2 / (1 + np.exp(-10 * (epoch / epochs))) - 1

        # Training batches
        for x, y, s in tqdm(train_loader):
            # Move inputs to device
            x = x.to(device)
            y = y.to(device).long().squeeze(-1)   # Task labels
            s = s.to(device).long().squeeze(-1)   # Domain (subject) labels

            optimizer.zero_grad()

            # Forward pass (with GRL)
            logits_task, logits_domain = model(x, lambd=lambd)

            # Losses:
            task_loss = criterion_task(logits_task, y)
            domain_loss = criterion_domain(logits_domain, s)

            # Weighted combination
            loss = task_loss + 0.5 * domain_loss

            # Backpropagation
            loss.backward()

            # Gradient clipping to stabilize training
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)

            # Update weights
            optimizer.step()

            # Track training performance
            running_loss += loss.item() * x.size(0)
            preds = torch.argmax(logits_task, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

        # Compute epoch-level training statistics
        train_loss = running_loss / total
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)


        # Validation phase
        model.eval()      # Disable dropout, batchnorm updates
        scheduler.step()  # Update learning rate

        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():   # No gradients during validation
            for x, y, s in tqdm(val_loader):
                x = x.to(device)
                y = y.to(device).long().squeeze(-1)

                # Domain loss is NOT used during validation, so lambd = 0
                logits_task, _ = model(x, lambd=0.0)

                loss = criterion_task(logits_task, y)

                val_loss += loss.item() * x.size(0)
                pred = torch.argmax(logits_task, dim=1)
                val_correct += (pred == y).sum().item()
                val_total += y.size(0)

        # Final validation epoch statistics
        val_loss /= val_total
        val_acc = val_correct / val_total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        # Print epoch summary
        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | λ={lambd:.3f}"
        )

        # Save best model checkpoint
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_model.pth")
            print("Saved new best model")

    print("Best Val Acc:", best_val_acc)


# **Load the data into training and val loaders**

In [ ]:
# Paths to training and validation HDF5 datasets
train_h5_path = '/kaggle/input/eeg-aad-task1/train_data.h5'
val_h5_path   = '/kaggle/input/eeg-aad-task1/val_data.h5'

# Load data from .h5 files (features, labels, subject IDs)
X_train, y_train, s_train = load_h5_dataset(train_h5_path)
X_val,   y_val,   s_val   = load_h5_dataset(val_h5_path)

# Subject ID arrays sometimes have an extra singleton dimension → remove it
s_train = s_train.squeeze()
s_val   = s_val.squeeze()

# Convert subject IDs to 0-based consecutive integer indices
# This ensures subject IDs are compatible with PyTorch operations.
unique_subjects = np.unique(np.concatenate([s_train, s_val]))     # Find all unique subject IDs
subject2idx = {sub: i for i, sub in enumerate(unique_subjects)}   # Map original ID → new index

# Apply conversion to both sets
s_train = np.array([subject2idx[s] for s in s_train])
s_val   = np.array([subject2idx[s] for s in s_val])

# Wrap datasets in PyTorch Dataset objects
# These handle indexing and formatting into PyTorch tensors.
train_dataset = CustomDatasets(X_train, y_train, s_train)
val_dataset   = CustomDatasets(X_val, y_val, s_val)

# (Optional sanity check) Print dataset shapes
print(f"Train data: {X_train.shape}, labels: {y_train.shape}")
print(f"Val data:   {X_val.shape}, labels: {y_val.shape}")

# Build DataLoaders for batching, shuffling, and parallel reading

train_loader = DataLoader(
    train_dataset,
    batch_size=128,   # Number of samples per batch
    shuffle=True,     # Shuffle training samples each epoch
    drop_last=True    # Ensures all batches are full-sized
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,    # No need to shuffle validation data
    drop_last=False   # Keep all samples
)


Train data: (171080, 32, 128), labels: (171080, 1)
Val data:   (26320, 32, 128), labels: (26320, 1)


# **Train the model**

In [ ]:
# create model
model = DARNet_LSTM_DANN(d_model = 8, num_subjects=len(unique_subjects))
# run the training
model = train_model(model, train_loader, val_loader, epochs=200, lr=1e-4, weight_decay = 3e-4, device="cuda")

100%|██████████| 206/206 [00:01<00:00, 132.17it/s]


Epoch 1/200 | Train Loss: 2.2350 | Train Acc: 0.6418 | Val Loss: 0.9926 | Val Acc: 0.5146 | λ=0.000
Saved new best model


100%|██████████| 206/206 [00:01<00:00, 132.16it/s]


Epoch 2/200 | Train Loss: 2.0821 | Train Acc: 0.6977 | Val Loss: 1.1881 | Val Acc: 0.4816 | λ=0.025


100%|██████████| 206/206 [00:01<00:00, 132.16it/s]


...

Epoch 198/200 | Train Loss: 1.6882 | Train Acc: 0.9775 | Val Loss: 3.1279 | Val Acc: 0.5297 | λ=1.000


100%|██████████| 206/206 [00:01<00:00, 134.45it/s]


Epoch 199/200 | Train Loss: 1.6978 | Train Acc: 0.9747 | Val Loss: 3.3241 | Val Acc: 0.5185 | λ=1.000


100%|██████████| 206/206 [00:01<00:00, 133.29it/s]

Epoch 200/200 | Train Loss: 1.6962 | Train Acc: 0.9747 | Val Loss: 3.1082 | Val Acc: 0.5320 | λ=1.000
Best Val Acc: 0.5609042553191489
